In this notebook, we analyze the scope diversity using the U-score and the R-score reported by Yang, Zhao, Luo, and co-workers (Angew. Chem. Int. Ed. 2026, 65, e2455429).

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..','..')))
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Code.benchmark import Benchmark
from Code.utils import obtain_full_covar_matrix
from sklearn.preprocessing import MinMaxScaler
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Draw
from rdkit import DataStructs
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import colorsys


# functions for U- and R-scores
from ScopeMap_Scores.evaluate import main as calculate_u_and_r_scores


# Doyle colors
doyle_colors = ["#CE4C6F", "#1561C2", "#188F9D","#C4ADA2","#515798", "#CB7D85", "#A9A9A9"]
# extension of palette with lighter and darker versions
def adjust_lightness(color, factor=1.2):
    """
    Function to make colors lighter (factor > 1) or darker (factor < 1).
    """
    r, g, b = mcolors.to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return mcolors.to_hex((r, g, b))

lighter = [adjust_lightness(c, 1.2) for c in doyle_colors]
darker  = [adjust_lightness(c, 0.7) for c in doyle_colors]
all_colors = doyle_colors + darker[::-1] + lighter[::-1] 

# Save the categorical colormap
cat_cmap = ListedColormap(all_colors, name="Doyle_cat")
plt.colormaps.register(cat_cmap)

# Define and save a continuous colormap
colors = [doyle_colors[1],"#FFFFFFD1",doyle_colors[0]]
cont_cmap = LinearSegmentedColormap.from_list("Doyle_cont", colors)
plt.colormaps.register(cont_cmap)
wdir = Path(".")


# General plt parameters
plt.rcParams.update({
    "axes.titlesize": 20,        # Subplot title
    "axes.labelsize": 16,        # X and Y labels
    "figure.titlesize": 24,      # Suptitle
    "xtick.labelsize": 14,       # X tick labels
    "ytick.labelsize": 14,       # Y tick labels
    "legend.fontsize": 14,       # Legend text
    "legend.title_fontsize": 14, # Legend titles
    "font.family": "Helvetica"   # Font
    })

In [2]:
# Read in the labelled dataset and scale it
df_labelled = pd.read_csv(f"./SB_dset.csv", index_col=0,header=0)
objectives = ["rate"]
wdir = Path(".")

### Recalculate the diversity scores with the U-Score and the R-Score

In [9]:
# save the search space smiles in a file (required input for the scores)
df_smiles = pd.DataFrame({"smiles":df_labelled.index})
df_smiles["smiles"] = df_smiles["smiles"].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x), isomericSmiles=True))
df_smiles.to_csv("ScopeMap_Scores/space_smiles.csv", index=False)

# calculate the U- and R-scores for the different acquisition functions and pruning settings
u_results = pd.DataFrame(np.nan, index=["Results"], columns=[])
r_results = pd.DataFrame(np.nan, index=["Results"], columns=[])
for acq in ["EI","Random","Greedy","Conv. selection","Explorative"]:
    if acq == "EI":
        acq_label = "balanced"
    elif acq == "Conv. selection":
        acq_label = "human-like-acq"
    else:
        acq_label = acq.lower()
    for pruning in [True,False]:
        if pruning:
            pruning_label = "_pruning"
            pruning_flag = "with"
        else:
            pruning_label = "_no-pruning"
            pruning_flag = "without"
        if acq == "Conv. selection":
            pruning_label = ""
        u_vals = []
        r_vals = []
        for filename in os.listdir(f"./Results_Data/scope_{acq_label}{pruning_label}/raw_data"):
            # open file and get the names of the selected samples
            df_filename = pd.read_csv(f"./Results_Data/scope_{acq_label}{pruning_label}/raw_data/{filename}",index_col=0,header=0)
            df_filename["eval_samples"] = df_filename["eval_samples"].apply(lambda x: [y.strip("'") for y in x[1:-1].split(', ')])
            samples = []
            for _,row in df_filename.iterrows():
                samples.extend(row["eval_samples"])
            samples = [Chem.MolToSmiles(Chem.MolFromSmiles(x), isomericSmiles=True) for x in samples]
            # save the selected samples in a file
            pd.DataFrame({"smiles": samples}).to_csv(f"ScopeMap_Scores/selected_smiles_{filename}", index=False)

            # calculate the scores
            u_val, r_val = calculate_u_and_r_scores([
                "--substrate-file", "ScopeMap_Scores/space_smiles.csv",
                "--experimental-file", f"ScopeMap_Scores/selected_smiles_{filename}",
            ])
            # clean up the temporary input file and also the generated fingerprint file
            os.remove(f"ScopeMap_Scores/selected_smiles_{filename}")
            os.remove(f"fp_spoc_morgan41024_Maccs_selected_smiles_{filename}")
            u_vals.append(u_val)
            r_vals.append(r_val)
        u_results.loc["Results", acq+pruning_label] = np.mean(u_vals)
        r_results.loc["Results", acq+pruning_label] = np.mean(r_vals)
u_results.rename(columns={"EI_pruning":"ScopeBO"},inplace=True)
r_results.rename(columns={"EI_pruning":"ScopeBO"},inplace=True)
label_dict = {col: col for col in u_results.columns}
for key,val in label_dict.items():
    if "_no-pruning" in val:
        label_dict[key] = val.split("_")[0]
    elif "pruning" in val:
        label_dict[key] = val.split("_")[0] + " (pruned)"
u_results.rename(columns=label_dict,inplace=True)
r_results.rename(columns=label_dict,inplace=True)
u_results.sort_values(by="Results",inplace=True, axis=1)
r_results.sort_values(by="Results",inplace=True, axis=1)

# remove the finger print file for the search space smiles and also the search space smiles file
os.remove("fp_spoc_morgan41024_Maccs_space_smiles.csv")
os.remove("ScopeMap_Scores/space_smiles.csv")

print("U-Scores:")
display(u_results)
print("R-Scores:")
display(r_results)

Arguments: Namespace(substrate_file='ScopeMap_Scores/space_smiles.csv', substrate_fp_file=None, experimental_file='ScopeMap_Scores/selected_smiles_27balanced_b3_V13_s15.csv', experimental_fp_file=None, distance_metric='euclidean', k_neighbors=5, random_seed=42, random_runs=5, not_feature_columns=['smiles'])
Starting Sampling Quality Evaluation

【Part 1: CVT and Kennard-Stone Sampling Evaluation】
--------------------------------------------------------------------------------

1. Reading data files...
   - Experimental data: (27, 1)
   - Sampling size set to experimental data length: 27
   - Fingerprint file 'fp_spoc_morgan41024_Maccs_space_smiles.csv' not found, generating...
   - Generated fingerprint file: fp_spoc_morgan41024_Maccs_space_smiles.csv
   - Combined complete space: (868, 1192)

2. Performing CVT sampling (sample size: 27)...
CVT algorithm converged, iterations: 12
   - CVT sampling completed, sample count: 27

3. Calculating CVT sampling evaluation metrics...
   - CVT sa

,Greedy,Random,Conv. selection,EI,Greedy (pruned),ScopeBO,Random (pruned),Explorative,Explorative (pruned)
Results,18.122944,40.085701,47.223047,50.537668,56.229125,66.747954,70.965406,72.249729,72.799065


R-Scores:


,Greedy,Conv. selection,Random,Greedy (pruned),Random (pruned),ScopeBO,Explorative,Explorative (pruned),EI
Results,-23.478589,-1.655877,4.002425,17.917338,43.864669,44.216498,45.201616,45.415681,81.074875


In [23]:
# add ranks to the results
u_results_ranked = u_results.rank(axis=1, method='min', ascending=False).astype(int)
r_results_ranked = r_results.rank(axis=1, method='min', ascending=False).astype(int)

print("U-Scores Ranked:")
display(u_results_ranked)
print("R-Scores Ranked:")
display(r_results_ranked)

U-Scores Ranked:


,Greedy,Random,Conv. selection,EI,Greedy (pruned),ScopeBO,Random (pruned),Explorative,Explorative (pruned)
U-Score,9,8,7,6,5,4,3,2,1


R-Scores Ranked:


,Greedy,Conv. selection,Random,Greedy (pruned),Random (pruned),ScopeBO,Explorative,Explorative (pruned),EI
R-Score,9,8,7,6,5,4,3,2,1


In [27]:
# combine u and r (plus ranks) into a single dataframe
u_results_combined = pd.concat([u_results, u_results_ranked], axis=0)
r_results_combined = pd.concat([r_results, r_results_ranked], axis=0)
results_combined = pd.concat([u_results_combined, r_results_combined], axis=0)
results_combined.index = ["U-Scores","U-Ranks","R-Scores","R-Ranks"]
results_combined.loc["U-Ranks"] = results_combined.loc["U-Ranks"].apply(lambda x: int(x))
results_combined.applymap(lambda x: round(x, 3) if isinstance(x, float) else x)

,Greedy,Random,Conv. selection,EI,Greedy (pruned),ScopeBO,Random (pruned),Explorative,Explorative (pruned)
U-Scores,18.123,40.086,47.223,50.538,56.229,66.748,70.965,72.250,72.799
U-Ranks,9.000,8.000,7.000,6.000,5.000,4.000,3.000,2.000,1.000
R-Scores,-23.479,4.002,-1.656,81.075,17.917,44.216,43.865,45.202,45.416
R-Ranks,9.000,7.000,8.000,1.000,6.000,4.000,5.000,3.000,2.000


In [28]:
results_combined.to_csv(f"U_and_R_Scores.csv", index=True, header=True)